#### Dependencies

In [1]:
import torch
import numpy as np
import pandas as pd
import yaml

#### Dataset

Variables

In [ ]:
# True: 20x20 MNIST, False: 28x28 MNIST
small_mnist = True
# True: Binarized Images, False: Grayscale Images 
binarize_images = True  
# True: Even distribution of samples, False: Original Mnist distribution 
evenly_partitioned = True
# Batch size
batch_size = 256

Dataset Transform

In [ ]:
# function to binarize an image, threshold is tunable 
def binarize(image, threshold=0.5):
    return (image > threshold).float()  

# define the transformation logic based on the toggle
if binarize_images:
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Lambda(lambda x: binarize(x))  # apply binarization if enabled
    ])
else:
    transform = transforms.Compose([
        transforms.ToTensor()  # just convert to tensor if not binarizing
    ])

In [ ]:
train_dataset = mnist_dataset.MNIST('./data-mnist', train=True, download=True, remove_border=small_mnist, transform=transform)
test_dataset = mnist_dataset.MNIST('./data-mnist', train=False, remove_border=small_mnist, transform=transform)

# drop_last = True means it will drop the last incomplete Batch
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, pin_memory=True, drop_last=True)

#### Model Hyperparameters

In [ ]:
# reads the CSV file into a DataFrame 
df = pd.read_csv("hyperparameters.csv")

# converts the DataFrame into a list of dictionaries
models = df.to_dict(orient="records")

# creates the YAML structure
yaml_structure = {"models": {}}

# rounds the number to the nearest multiple of the output size
# DiffLogic expects output layer to a multiple of the # of classes
def round_to_nearest_multiple(value, multiple):
    return multiple * round(value / multiple)

# populates the YAML structure with models
for i, model in enumerate(models, start=1):
    # zero-padding the model names to 3 digits 
    model_name = f"model_{str(i).zfill(3)}"
    layers_config = {}
    
    for layer in range(1, model["H"] + 1):
        # zero-padding the layer names to 3 digits
        layer_name = f"LogicLayer{str(layer).zfill(3)}"
        
        # adjusts the in_dim to the nearest multiple of 10
        in_dim = 400 if layer == 1 else round_to_nearest_multiple(model["W"], 10)
        
        # adjusts the out_dim to the nearest multiple of 10
        out_dim = round_to_nearest_multiple(model["W"], 10)
        
        layers_config[layer_name] = {
            "in_dim": in_dim,
            "out_dim": out_dim,
            "device": "cuda",
            "implementation": "cuda",
            "connections": "random",
            "grad_factor": 1, # other values can be tried
        }
    
    yaml_structure["models"][model_name] = {
        "input_dim": 400, # 20x20 MNIST Image
        "output_size": 10, # for MNIST classification
        "tau": model["tau"],
        "learning_rate": model["lr"],
        "layers_config": layers_config,
    }

# saves to a YAML file
with open("mnist_config.yaml", "w") as file:
    yaml.dump(yaml_structure, file, default_flow_style=False)

print("YAML file 'mnist_config.yaml' generated successfully.")

#### Model Definition

#### Model Instantiation

#### Model Training

#### Model Inference

#### Extracting Model Gates and Connections

In [ ]:
for idx, layer in enumerate(model.diff_logic_model.logic_layers):
    if isinstance(layer, LogicLayer):
        # extracts connections
        connections = layer.indices 
        print(f"Layer {idx} connections: {connections}")
        
        # extracts distribution of learned parameters (gates)
        print(f"Layer {idx} learned gates:")
        for name, param in layer.named_parameters():
            print(f"  Parameter '{name}':")
            print(param.data)  

#### Verilog Conversion

In [ ]:
# Logic gate to Verilog expression mapping
logic_gate_verilog = {
    "0": "1'b0",
    "A∧B": "{a} & {b}",
    "¬(A⇒B)": "{a} & ~{b}",
    "A": "{a}",
    "¬(B⇒A)": "{b} & ~{a}",
    "B": "{b}",
    "A⊕B": "{a} ^ {b}",
    "A∨B": "{a} | {b}",
    "¬(A∨B)": "~({a} | {b})",
    "¬(A⊕B)": "~({a} ^ {b})",
    "¬B": "~{b}",
    "B⇒A": "~{b} | {a}",
    "¬A": "~{a}",
    "A⇒B": "~{a} | {b}",
    "¬(A∧B)": "~({a} & {b})",
    "1": "1'b1"
}

# Assuming `model` and `logic_layer` have already been defined and populated
layer_index = 0
logic_layer = model.logic_layers[layer_index]

input_indices = logic_layer.indices[0].cpu().numpy()  # First input indices
output_indices = logic_layer.indices[1].cpu().numpy()  # Second input indices
neuron_gates = [torch.argmax(logic_layer.weights[neuron]).item() for neuron in range(logic_layer.weights.size()[0])]
connections = {i: (input_indices[i], output_indices[i]) for i in range(len(neuron_gates))}

def generate_verilog(neuron_gates, connections, filename="logic_network.v"):
    with open(filename, 'w') as file:
        file.write("module logic_network(\n")
        file.write("    input wire [{}:0] inputs,\n".format(max(max(connections.keys()), max(max(con) for con in connections.values()))))
        file.write("    output wire [{}:0] outputs\n".format(len(neuron_gates) - 1))
        file.write(");\n\n")

        # Declare wires for internal connections
        for neuron_id in connections:
            a_idx, b_idx = connections[neuron_id]
            gate = logic_gate_verilog[logic_operations[neuron_gates[neuron_id]]].format(a=f"inputs[{a_idx}]", b=f"inputs[{b_idx}]")
            file.write(f"    assign outputs[{neuron_id}] = {gate};\n")

        file.write("endmodule\n")

# Generate Verilog file
generate_verilog(neuron_gates, connections)

In [ ]:
# Logic gate to Verilog expression mapping
logic_gate_verilog = {
    "0": "1'b0",
    "A∧B": "({a} & {b})",
    "¬(A⇒B)": "({a} & ~{b})",
    "A": "{a}",
    "¬(B⇒A)": "({b} & ~{a})",
    "B": "{b}",
    "A⊕B": "({a} ^ {b})",
    "A∨B": "({a} | {b})",
    "¬(A∨B)": "~({a} | {b})",
    "¬(A⊕B)": "~({a} ^ {b})",
    "¬B": "~{b}",
    "B⇒A": "(~{b} | {a})",
    "¬A": "~{a}",
    "A⇒B": "(~{a} | {b})",
    "¬(A∧B)": "~({a} & {b})",
    "1": "1'b1"
}

# Assuming 'model' is your trained Model instance
logic_layers = model.diff_logic_model.logic_layers

# Initialize a mapping from indices to signal names
signal_mapping = {}
signals = []  # List of all signal names

# For inputs: indices 0 to num_inputs - 1
num_inputs = model.diff_logic_model.flatten.num_features  # Adjust based on your model's input dimension
for idx in range(num_inputs):
    signal_name = f"input_{idx}"
    signal_mapping[idx] = signal_name
    signals.append(signal_name)

current_index = num_inputs  # Next available index for outputs

wire_definitions = []
assign_statements = []

# Logic operations mapping (adjust according to your implementation)
logic_operations = {
    0: "0",
    1: "A∧B",
    2: "¬(A⇒B)",
    3: "A",
    4: "¬(B⇒A)",
    5: "B",
    6: "A⊕B",
    7: "A∨B",
    8: "¬(A∨B)",
    9: "¬(A⊕B)",
    10: "¬B",
    11: "B⇒A",
    12: "¬A",
    13: "A⇒B",
    14: "¬(A∧B)",
    15: "1"
}

# Iterate over each layer in the model
for layer in logic_layers:
    if isinstance(layer, LogicLayer):
        # Get connections and gates
        # Assuming layer.indices is a list or tensor of input index pairs for each neuron
        connections = layer.indices.cpu().numpy()  # Shape: [num_neurons, num_inputs_per_neuron]
        neuron_gates = [torch.argmax(layer.weights[neuron]).item() for neuron in range(layer.weights.size(0))]
        
        num_neurons = len(neuron_gates)
        for neuron_id in range(num_neurons):
            gate_idx = neuron_gates[neuron_id]
            gate = logic_operations[gate_idx]
            
            # Get the input indices for this neuron
            input_indices = connections[neuron_id]
            a_idx, b_idx = input_indices  # Assuming two inputs per neuron
            
            # Get the input signals
            a_signal = signal_mapping[a_idx]
            b_signal = signal_mapping[b_idx]
            
            # Generate a unique signal name for the output of this neuron
            output_signal = f"wire_{current_index}"
            signal_mapping[current_index] = output_signal
            signals.append(output_signal)
            current_index += 1
            
            # Define the wire
            wire_definitions.append(f"wire {output_signal};")
            
            # Get the Verilog expression for the gate
            verilog_expr = logic_gate_verilog[gate].format(a=a_signal, b=b_signal)
            assign_statements.append(f"assign {output_signal} = {verilog_expr};")
